# Evaluate shape mappers

Evaluate every mapper in `MAPPER_NAMES` on the same deterministic set of eight toys4k pairs per shape. Metrics are computed per pair and then averaged across pairs. Subdivision metrics are reported separately for every decoder stage. This notebook only displays an in-memory DataFrame.

In [ ]:
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
import trellis2.models as trellis2_models
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from trellis2.modules.sparse import SparseTensor

from dataset import DirectFileLoadDataset, MultiScaleMixedPairSampler, build_catalog, pair_collate
from symtrellis.geometry import t_abs2grid
from symtrellis.mapper import from_pretrained


In [ ]:
MAPPER_NAMES = [
    "trellis2/shape/swin3d/legacy",
    "trellis2/shape/neighbor_graph/finetune",
]

TOYS4K_DATA_DIR = Path(
    "/mnt/scratch/trellis500k/toys4k/trellis2/multi_slats/"
    "shape_enc_next_dc_f16c32_fp16_shapelatentres_32_ovoxres_512_s3_r6_p3"
)
DECODER_PRETRAINED_PATH = "microsoft/TRELLIS.2-4B/ckpts/shape_dec_next_dc_f16c32_fp16"

PAIRS_PER_SHAPE = 8
EVAL_BATCH_SIZE = 4
EVAL_SEED = 42
NUM_WORKERS = 8
PIN_MEMORY = True
PREFETCH_FACTOR = 2
PERSISTENT_WORKERS = True
DEVICE = torch.device("cuda")

GRID_SIZE = 32
NUM_SCALES = 3
NUM_ROTS = 6
NUM_PERTS = 3
SAME_ROT_DIFF_PERT_RATIO = 0.2
DECODER_RESOLUTION = 512


## Fixed evaluation pairs

The full toys4k catalog is evaluated without a val/test split. Every mapper receives the same pair indices in the same order.

In [ ]:
shape_paths = build_catalog([TOYS4K_DATA_DIR], seed=EVAL_SEED)
eval_dataset = DirectFileLoadDataset(
    shape_paths=shape_paths,
    grid_size=GRID_SIZE,
    num_scale=NUM_SCALES,
    num_rots=NUM_ROTS,
    num_perts=NUM_PERTS,
)
index_sampler = MultiScaleMixedPairSampler(
    num_shapes=eval_dataset.num_shapes,
    num_scale=eval_dataset.num_scale,
    num_rots=eval_dataset.num_rots,
    num_perts=eval_dataset.num_perts,
    batch_size=1,
    num_batch_per_epoch=eval_dataset.num_shapes * PAIRS_PER_SHAPE,
    rank=0,
    world_size=1,
    seed=EVAL_SEED,
    same_rot_diff_pert_ratio=SAME_ROT_DIFF_PERT_RATIO,
)
pair_indices = [batch[0] for batch in index_sampler]
eval_loader = DataLoader(
    eval_dataset,
    batch_size=EVAL_BATCH_SIZE,
    sampler=pair_indices,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    collate_fn=pair_collate,
)

print(f"shapes: {eval_dataset.num_shapes:,}")
print(f"pairs:  {len(pair_indices):,}")
print(f"batches: {len(eval_loader):,}")


In [ ]:
decoder = trellis2_models.from_pretrained(DECODER_PRETRAINED_PATH)
decoder.convert_to_fp32()
decoder.use_fp16 = False
decoder.dtype = torch.float32
decoder.set_resolution(DECODER_RESOLUTION)
decoder.float().eval().to(DEVICE)
for parameter in decoder.parameters():
    parameter.requires_grad_(False)


In [ ]:
def decode_shape_features(decoder, latent, guide_subdivisions=None):
    h = decoder.from_latent(latent)
    h = h.type(decoder.dtype)
    subdivision_logits = []

    for stage_index, blocks in enumerate(decoder.blocks):
        for block_index, block in enumerate(blocks):
            is_subdivision_block = (
                stage_index < len(decoder.blocks) - 1
                and block_index == len(blocks) - 1
            )
            if not is_subdivision_block:
                h = block(h)
                continue

            if guide_subdivisions is None:
                h, subdivision = block(h)
                subdivision_logits.append(subdivision)
                continue

            subdivision = block.to_subdiv(h)
            guide = guide_subdivisions[len(subdivision_logits)]
            guide_mask = guide.replace(guide.feats > 0)
            residual = h
            h = h.replace(block.norm1(h.feats))
            h = h.replace(F.silu(h.feats))
            h = block.conv1(h)
            h = block.updown(h, guide_mask)
            residual = block.updown(residual, guide_mask)
            h = h.replace(block.norm2(h.feats))
            h = h.replace(F.silu(h.feats))
            h = block.conv2(h)
            h = h + block.skip_connection(residual)
            subdivision_logits.append(subdivision)

    h = h.type(latent.dtype)
    h = h.replace(F.layer_norm(h.feats, h.feats.shape[-1:]))
    return decoder.output_layer(h), subdivision_logits


@torch.no_grad()
def evaluate_shape_mapper(model, loader, decoder):
    metric_sums = {}
    num_evaluated_samples = 0

    model.eval()
    decoder_dtype = next(decoder.parameters()).dtype
    for batch in tqdm(loader, desc="evaluate", leave=False):
        batch = {name: tensor.to(DEVICE, non_blocking=True) for name, tensor in batch.items()}
        t_grid = t_abs2grid(
            t_abs=batch["t_dst2src"],
            O=batch["O_dst2src"],
            grid_size=GRID_SIZE,
        )
        coeff = model(
            coords_src=batch["coords_src"],
            coords_dst=batch["coords_dst"],
            O_dst2src=batch["O_dst2src"],
            t_dst2src=t_grid,
            s_dst2src=batch["s_dst2src"],
        )
        prediction = coeff.apply(batch["feats_src"].to(dtype=coeff.dtype)).float()
        target = batch["feats_dst"].float()
        destination_batch_ids = batch["coords_dst"][:, 0].long()

        target_slat = SparseTensor(
            feats=target.to(dtype=decoder_dtype),
            coords=batch["coords_dst"],
        )
        prediction_slat = SparseTensor(
            feats=prediction.to(dtype=decoder_dtype),
            coords=batch["coords_dst"],
        )
        target_output, target_subdivisions = decode_shape_features(decoder, target_slat)
        prediction_output, prediction_subdivisions = decode_shape_features(
            decoder,
            prediction_slat,
            target_subdivisions,
        )

        batch_size = batch["O_dst2src"].shape[0]
        output_batch_ids = target_output.coords[:, 0].long()
        subdivision_batch_ids = [subdivision.coords[:, 0].long() for subdivision in target_subdivisions]

        for sample_id in range(batch_size):
            feature_mask = destination_batch_ids == sample_id
            sample_prediction = prediction[feature_mask]
            sample_target = target[feature_mask]
            feature_diff = sample_prediction - sample_target
            per_row_cosine = F.cosine_similarity(sample_prediction, sample_target, dim=1)
            feature_cosine = per_row_cosine.mean()
            sample_metrics = {
                "feature_l1": feature_diff.abs().mean(dim=1).mean(),
                "feature_l2": feature_diff.square().mean(dim=1).mean(),
                "feature_cosine": feature_cosine,
                "feature_cosine_distance": 1.0 - feature_cosine,
            }

            output_mask = output_batch_ids == sample_id
            output_diff = (
                prediction_output.feats[output_mask].float()
                - target_output.feats[output_mask].float()
            )
            sample_metrics["decoder_output_l2"] = output_diff.square().mean()

            for stage_index, (prediction_subdivision, target_subdivision) in enumerate(
                zip(prediction_subdivisions, target_subdivisions)
            ):
                subdivision_mask = subdivision_batch_ids[stage_index] == sample_id
                prediction_logits = prediction_subdivision.feats[subdivision_mask].float()
                target_logits = target_subdivision.feats[subdivision_mask].float()
                sample_metrics[f"decoder_subdivision_l2_stage_{stage_index}"] = (
                    prediction_logits - target_logits
                ).square().mean()
                prediction_occupied = prediction_logits > 0
                target_occupied = target_logits > 0
                intersection = (prediction_occupied & target_occupied).sum().float()
                union = (prediction_occupied | target_occupied).sum().float()
                sample_metrics[f"decoder_subdivision_iou_stage_{stage_index}"] = (
                    intersection / union.clamp_min(1.0)
                )

            for name, value in sample_metrics.items():
                metric_sums[name] = metric_sums.get(name, value.new_zeros((), dtype=torch.float64)) + value.double()
            num_evaluated_samples += 1

    result = {"num_evaluated_samples": num_evaluated_samples}
    result.update({name: (value / num_evaluated_samples).item() for name, value in metric_sums.items()})
    return result


In [ ]:
rows = []
for model_name in tqdm(MAPPER_NAMES, desc="models"):
    print(f"Evaluating {model_name}")
    model = from_pretrained(model_name, device=DEVICE).eval()
    metrics = evaluate_shape_mapper(model, eval_loader, decoder)
    rows.append({"model_name": model_name, **metrics})
    del model
    torch.cuda.empty_cache()

evaluation_df = pd.DataFrame(rows)
evaluation_df
